In [1]:
import pandas as pd
from datetime import datetime

from pandas import DataFrame
from games import read_games_file, add_features_to_games, filter_games
from seasons_info import make_season_info
from teams_records import (
    calculate_record,
    calculate_away_record,
    calculate_home_record,
    get_records,
    get_home_records,
    get_away_records,
    make_east_west_record,
)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)


In [2]:
games = read_games_file()
games: DataFrame = add_features_to_games(games)

# Test different cutting dates with DVC
GAMETYPES = ["Regular Season", "NBA Emirates Cup"]
START_DATE = "1980-07-01"
games_filtered: DataFrame = filter_games(games, START_DATE, GAMETYPES)
games_filtered = games_filtered.drop(
    columns=["gameType", "gameSubLabel", "gameLabel", "seriesGameNumber"]
).copy()

# Season info
season_info = make_season_info(games_filtered)

# Filter by one season for testing purposes
SEASON = "2023/24"
season_info = season_info[season_info["season"] == SEASON]
season_start = list(season_info["season_start"])[0]
season_end = list(season_info["season_end"])[0]

games_filtered = games_filtered[games_filtered["season"] == SEASON]  # type: ignore
games_filtered = games_filtered.drop(columns=["season"]).copy()

# Teams Records info
home_games_record = calculate_home_record(games_filtered)
away_games_record = calculate_away_record(games_filtered)

teams_record = calculate_record(
    pd.concat([home_games_record, away_games_record], ignore_index=True)
)

games_filtered = get_records(games_filtered, teams_record)
games_filtered = get_home_records(
    games_filtered, home_games_record
)
games_filtered = get_away_records(
    games_filtered, away_games_record
)

games_filtered = games_filtered.rename(
    columns={
        "total_wins": "total_wins_HT",
        "total_losses": "total_losses_HT",
        "record": "record_HT",
        "record_L5": "record_L5_HT",
        "pts_diff_avg": "pts_diff_avg_HT",
        "pts_diff_avg_L5": "pts_diff_avg_L5_HT",
        "games_played": "games_played_HT",
    }
)

# East vs West Record
east_west_record_by_date = make_east_west_record(games, season_start, season_end)
games_filtered = games_filtered.merge(
    east_west_record_by_date[
        ["gameDateOnlyStr", "east_wins_pct_lag1"]
    ],
    how="left",
    on=["gameDateOnlyStr"],
)

/Users/felipeformenti/dev/nba_bets/games.py:9: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, parse_dates=["gameDate"])


In [6]:
games_filtered.head(5)


,gameId,gameDate,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,attendance,arenaId,winnerteamCity,gameDateOnlyStr,pts_diff,winner_home_bool,winner_away_bool,hometeamConference,awayteamConference,winnerteamConference,total_wins_HT,total_losses_HT,record_HT,record_L5_HT,pts_diff_avg_HT,pts_diff_avg_L5_HT,games_played_HT,total_wins_VT,total_losses_VT,record_VT,record_L5_VT,pts_diff_avg_VT,pts_diff_avg_L5_VT,games_played_VT,record_HT_at_home,record_L5_HT_at_home,pts_diff_avg_HT_at_home,pts_diff_avg_L5_HT_at_home,record_VT_at_away,record_L5_VT_at_away,pts_diff_avg_VT_at_away,pts_diff_avg_L5_VT_at_away,east_wins_pct_lag1
0,22301193,2024-04-14 15:30:00,Memphis,Grizzlies,1610612763,Denver,Nuggets,1610612743,111,126,1610612743,17544.0,242,Denver,2024-04-14,-15,0,1,West,West,West,27.0,54.0,0.333333,0.2,-0.209877,-1.6,81.0,56.0,25.0,0.691358,0.6,4.765432,5.6,81.0,0.225000,0.2,-7.200000,-6.4,0.575000,0.6,-0.375000,-3.8,NaN
1,22301194,2024-04-14 15:30:00,Minnesota,Timberwolves,1610612750,Phoenix,Suns,1610612756,106,125,1610612756,18024.0,61,Phoenix,2024-04-14,-19,0,1,West,West,West,56.0,25.0,0.691358,0.6,2.049383,4.2,81.0,48.0,33.0,0.592593,0.6,0.172840,-5.6,81.0,0.750000,0.8,8.925000,11.8,0.575000,0.8,-2.725000,-2.4,NaN
2,22301195,2024-04-14 15:30:00,New Orleans,Pelicans,1610612740,Los Angeles,Lakers,1610612747,108,124,1610612747,18633.0,160,Los Angeles,2024-04-14,-16,0,1,West,West,West,49.0,32.0,0.604938,0.8,-0.864198,-7.4,81.0,46.0,35.0,0.567901,0.6,4.185185,-2.6,81.0,0.538462,0.2,3.948718,-5.8,0.461538,0.8,3.923077,-3.6,NaN
3,22301196,2024-04-14 15:30:00,Oklahoma City,Thunder,1610612760,Dallas,Mavericks,1610612742,135,86,1610612760,18203.0,1000052,Oklahoma City,2024-04-14,49,1,0,West,West,West,56.0,25.0,0.691358,0.8,5.000000,14.8,81.0,50.0,31.0,0.617284,0.8,0.839506,-10.0,81.0,0.800000,0.8,12.050000,16.4,0.625000,0.8,-2.025000,-12.6,NaN
4,22301197,2024-04-14 15:30:00,San Antonio,Spurs,1610612759,Detroit,Pistons,1610612765,123,95,1610612759,18516.0,1000118,San Antonio,2024-04-14,28,1,0,West,East,West,21.0,60.0,0.259259,0.6,2.024691,3.0,81.0,14.0,67.0,0.172840,0.2,0.950617,1.2,81.0,0.275000,0.6,-4.950000,-0.8,0.170732,0.2,9.707317,7.2,NaN


In [ ]:
east_west_record_by_date


In [ ]:
games_filtered.to_csv(
    "data/output/games.csv",
    index=False,
)

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("hugging_face_secret"))
api.upload_folder(
    folder_path="/data/output/games.csv",
    repo_id="fformenti/games",
    repo_type="dataset",
)


In [ ]:
home_games.head(10)


In [ ]:
# rested_days[rested_days["teamId"] == 1610612743].head(20)

In [ ]:
# set(rested_days.teamId)

In [ ]:
games.info()
